In [ ]:
# data cleaning
import html
import re
import subprocess
import sys
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
INPUT_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
N_FOLDS = 5
SEED = 42

# ---------------------------------------------------------------------------
# DEPENDENCIES (auto-install iterstrat on Kaggle)
# ---------------------------------------------------------------------------
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold  # noqa
    HAS_ITERSTRAT = True
except ImportError:
    print("Installing iterative-stratification ...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "iterative-stratification"]
    )
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold  # noqa
        HAS_ITERSTRAT = True
    except ImportError:
        HAS_ITERSTRAT = False


# ---------------------------------------------------------------------------
# CONTRACTIONS (comprehensive dictionary, longest-first matching)
# ---------------------------------------------------------------------------
CONTRACTIONS = {
    "ain't": "is not", "aren't": "are not", "can't": "cannot",
    "couldn't": "could not", "didn't": "did not", "doesn't": "does not",
    "don't": "do not", "hadn't": "had not", "hasn't": "has not",
    "haven't": "have not", "he'd": "he would", "he'll": "he will",
    "he's": "he is", "i'd": "i would", "i'll": "i will", "i'm": "i am",
    "i've": "i have", "isn't": "is not", "it's": "it is", "let's": "let us",
    "mustn't": "must not", "shan't": "shall not", "she'd": "she would",
    "she'll": "she will", "she's": "she is", "shouldn't": "should not",
    "that's": "that is", "there's": "there is", "they'd": "they would",
    "they'll": "they will", "they're": "they are", "they've": "they have",
    "we'd": "we would", "we're": "we are", "we've": "we have",
    "weren't": "were not", "what's": "what is", "where's": "where is",
    "who's": "who is", "won't": "will not", "wouldn't": "would not",
    "you'd": "you would", "you'll": "you will", "you're": "you are",
    "you've": "you have", "y'all": "you all",
    "wanna": "want to", "gonna": "going to", "gotta": "got to",
}
CONTRACTION_RE = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in sorted(CONTRACTIONS, key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)

# ---------------------------------------------------------------------------
# STOPWORDS (with KEEP_WORDS carveout for toxicity signal words)
# ---------------------------------------------------------------------------
KEEP_WORDS = {
    # 2nd-person address — very strong toxicity signal
    "you", "your", "yours", "yourself", "u", "ur", "youre",
    # negation — flips meaning
    "no", "not", "nor", "never", "none", "nothing", "nobody",
    "cant", "cannot", "dont", "doesnt", "didnt", "wont", "shouldnt",
    "wouldnt", "couldnt", "isnt", "arent", "wasnt", "werent", "aint",
    # intensifiers
    "why", "who", "very", "too", "so", "just", "only", "own", "same",
}
BASE_STOPWORDS = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "he", "him",
    "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself",
    "they", "them", "their", "theirs", "themselves", "what", "which", "whom",
    "this", "that", "these", "those", "am", "is", "are", "was", "were", "be",
    "been", "being", "have", "has", "had", "having", "do", "does", "did",
    "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as",
    "until", "while", "of", "at", "by", "for", "with", "about", "against",
    "between", "into", "through", "during", "before", "after", "above",
    "below", "to", "from", "up", "down", "in", "out", "on", "off", "over",
    "under", "again", "further", "then", "once", "here", "there", "when",
    "where", "how", "all", "any", "both", "each", "few", "more", "most",
    "other", "some", "such", "than", "s", "t", "can", "will", "should",
    "now", "d", "ll", "m", "o", "re", "ve", "y",
}
STOPWORDS = BASE_STOPWORDS - KEEP_WORDS


# ---------------------------------------------------------------------------
# LEET-SPEAK MAP  (for deobfuscating masked profanity)
# ---------------------------------------------------------------------------
LEET_MAP = str.maketrans({
    "@": "a", "$": "s", "0": "o", "1": "i", "3": "e",
    "4": "a", "5": "s", "7": "t", "|": "i", "+": "t",
})


# ---------------------------------------------------------------------------
# REGEX PATTERNS (compiled once for speed)
# ---------------------------------------------------------------------------
# Wikipedia namespace / markup
RE_WIKI_USER = re.compile(
    r"\[\[\s*(?:user|user talk|special:contributions)\s*:[^\]]*\]\]", re.IGNORECASE
)
RE_WIKI_IMG = re.compile(
    r"\[\[\s*(?:image|file|category)\s*:[^\]]*\]\]", re.IGNORECASE
)
RE_WIKI_LINK = re.compile(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]")
RE_EXT_LINK = re.compile(r"\[(?:https?://\S+)\s*([^\]]*)\]")
RE_TEMPLATE = re.compile(r"\{\{[^{}]*\}\}")
RE_HEADING = re.compile(r"={2,}\s*([^=]+?)\s*={2,}")
RE_WIKI_EMPH = re.compile(r"'{2,5}")
RE_TALKMARK = re.compile(r"^[:*#]+", re.MULTILINE)
RE_TIMESTAMP = re.compile(
    r"\d{1,2}:\d{2},\s*\d{1,2}\s+\w+\s+\d{4}\s*\(?UTC\)?", re.IGNORECASE
)
RE_UTC = re.compile(r"\(\s*UTC\s*\)", re.IGNORECASE)

# HTML
RE_HTML_TAG = re.compile(r"<[^>]{1,200}>")

# Web noise
RE_URL = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
RE_EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
RE_IP = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

# Character hygiene
RE_CONTROL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
RE_NEWLINE = re.compile(r"[\r\n\t\u000b\u000c\u0085\u2028\u2029]+")

# Repetition (letters vs punctuation handled separately!)
RE_REPEAT_CHAR = re.compile(r"([A-Za-z])\1{2,}")     # fuuuck -> fuuck
RE_REPEAT_PUNC = re.compile(r"([!?.,])\1{3,}")       # !!!!! -> !!!
RE_WS = re.compile(r"\s+")
RE_NON_ALPHA = re.compile(r"[^a-z\s]")

# Obfuscation
RE_SPACED_WORD = re.compile(r"\b(?:[a-zA-Z]\s){2,}[a-zA-Z]\b")       # f u c k -> fuck
RE_DOTTED_WORD = re.compile(r"\b(?:[a-zA-Z][.\-_*]){2,}[a-zA-Z]\b")  # f.u.c.k -> fuck
RE_MASKED = re.compile(r"\b[a-zA-Z]+[*@$#!]{1,4}[a-zA-Z]*\b")       # f**k, sh1t -> after leet


# ---------------------------------------------------------------------------
# CLEANING PRIMITIVES
# ---------------------------------------------------------------------------
def normalize_unicode(text: str) -> str:
    """NFKC compat-decompose (Ｆuck -> Fuck) + drop control characters."""
    text = unicodedata.normalize("NFKC", text)
    return RE_CONTROL.sub(" ", text)


def strip_accents(text: str) -> str:
    """café -> cafe (used for classical only)."""
    return "".join(
        c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c)
    )


def strip_wiki_markup(text: str) -> str:
    """Remove Wikipedia Talk-page artifacts specific to this dataset."""
    text = RE_WIKI_USER.sub(" ", text)     # drop [[User:...]] entirely
    text = RE_WIKI_IMG.sub(" ", text)      # drop [[Image/File/Category:...]]
    text = RE_TEMPLATE.sub(" ", text)      # drop {{template}}
    text = RE_HEADING.sub(r" \1 ", text)   # keep heading text
    text = RE_WIKI_LINK.sub(r" \1 ", text) # keep display text of wiki links
    text = RE_EXT_LINK.sub(r" \1 ", text)  # keep display text of [url text]
    text = RE_WIKI_EMPH.sub(" ", text)     # drop '', ''', ''''', ...
    text = RE_TALKMARK.sub(" ", text)      # drop ^:, ^*, ^# talk indents
    return text


def strip_web_noise(text: str) -> str:
    """Remove URLs, emails, and IPv4 addresses."""
    text = RE_URL.sub(" ", text)
    text = RE_EMAIL.sub(" ", text)
    text = RE_IP.sub(" ", text)
    return text


def deobfuscate(text: str) -> str:
    """
    Undo common toxicity-obfuscation tricks:
      f u c k       -> fuck        (spaced letters)
      f.u.c.k       -> fuck        (dotted)
      f-u-c-k       -> fuck        (hyphenated)
      f**k / sh1t   -> fuck / shit (leetspeak + mask chars)
    """
    text = RE_SPACED_WORD.sub(lambda m: m.group(0).replace(" ", ""), text)
    text = RE_DOTTED_WORD.sub(lambda m: re.sub(r"[.\-_*]", "", m.group(0)), text)
    text = RE_MASKED.sub(
        lambda m: re.sub(r"[*#!]", "", m.group(0).translate(LEET_MAP)), text
    )
    return text


def expand_contractions(text: str) -> str:
    """can't -> cannot, you're -> you are (case-insensitive dict lookup)."""
    return CONTRACTION_RE.sub(lambda m: CONTRACTIONS[m.group(0).lower()], text)


# ---------------------------------------------------------------------------
# TWO-LEVEL CLEANER
# ---------------------------------------------------------------------------
def clean_light(text: str) -> str:
    """
    LIGHT cleaning for pretrained transformers (BERT / DistilBERT / RoBERTa).
    Keeps casing, punctuation, and contractions — the tokenizer needs them.
    """
    if not isinstance(text, str):
        return ""
    text = html.unescape(html.unescape(text))     # double-encoded entities exist
    text = normalize_unicode(text)
    text = RE_HTML_TAG.sub(" ", text)
    text = strip_wiki_markup(text)
    text = RE_TIMESTAMP.sub(" ", text)
    text = RE_UTC.sub(" ", text)
    text = strip_web_noise(text)
    text = RE_NEWLINE.sub(" ", text)
    text = RE_REPEAT_CHAR.sub(r"\1\1", text)       # letters: keep 2
    text = RE_REPEAT_PUNC.sub(r"\1\1\1", text)     # punct:   keep 3
    return RE_WS.sub(" ", text).strip()


def clean_heavy(text: str, drop_stopwords: bool = True) -> str:
    """
    HEAVY cleaning for bag-of-words models (TF-IDF + LogReg / SVM / NB).
    Runs on top of LIGHT output. Aggressive: lowercase, accent-strip,
    deobfuscate, expand contractions, letters only, optional stopword drop.
    """
    if not text:
        return ""
    text = strip_accents(text.lower())
    text = deobfuscate(text)
    text = expand_contractions(text)
    text = RE_NON_ALPHA.sub(" ", text)
    tokens = [t for t in text.split() if len(t) > 1]
    if drop_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]
    return " ".join(tokens)


# ---------------------------------------------------------------------------
# FEATURE ENGINEERING (extra columns for classical models)
# ---------------------------------------------------------------------------
def build_features(raw: pd.Series) -> pd.DataFrame:
    """
    Hand-crafted numeric features from RAW text. These are strong signals for
    classical models — you feed them alongside TF-IDF (hstack).
    """
    s = raw.fillna("")
    words = s.str.split()
    n_words = words.str.len().replace(0, np.nan)
    n_chars = s.str.len().replace(0, np.nan)

    f = pd.DataFrame(index=s.index)
    f["n_chars"] = s.str.len()
    f["n_words"] = words.str.len()
    f["n_unique_words"] = words.apply(lambda w: len(set(w)) if isinstance(w, list) else 0)
    f["unique_word_ratio"] = (f["n_unique_words"] / n_words).fillna(0)
    f["mean_word_len"] = (s.str.replace(r"\s", "", regex=True).str.len() / n_words).fillna(0)
    f["caps_ratio"] = (s.str.count(r"[A-Z]") / n_chars).fillna(0)
    f["n_exclaim"] = s.str.count(r"!")
    f["n_question"] = s.str.count(r"\?")
    f["punct_ratio"] = (s.str.count(r"[^\w\s]") / n_chars).fillna(0)
    f["n_newlines"] = s.str.count("\n")
    f["n_urls"] = s.str.count(RE_URL)
    f["n_ips"] = s.str.count(RE_IP)
    f["n_you"] = s.str.count(r"(?i)\b(you|your|u|ur)\b")   # 2nd-person address
    f["n_masked_words"] = s.str.count(r"\b[a-zA-Z]+[*@$#]{1,4}[a-zA-Z]*\b")
    f["has_shouting"] = (f["caps_ratio"] > 0.30).astype(int)
    return f


# ---------------------------------------------------------------------------
# QUALITY AUDIT
# ---------------------------------------------------------------------------
def audit(df: pd.DataFrame, name: str) -> None:
    print(f"\n--- audit: {name} ---")
    print(f"rows              : {len(df):,}")
    print(f"null comment_text : {df['comment_text'].isna().sum()}")
    print(f"empty/whitespace  : {(df['comment_text'].fillna('').str.strip() == '').sum()}")
    print(f"duplicate ids     : {df['id'].duplicated().sum()}")
    print(f"duplicate texts   : {df['comment_text'].duplicated().sum()}")
    present = [c for c in LABELS if c in df.columns]
    if present:
        for c in present:
            n = int(df[c].sum())
            print(f"  {c:<14} {n:>7,}  ({n / len(df) * 100:5.2f}%)")
        clean_rows = int((df[present].sum(axis=1) == 0).sum())
        print(f"  {'no label':<14} {clean_rows:>7,}  ({clean_rows / len(df) * 100:5.2f}%)")


# ---------------------------------------------------------------------------
# PROCESS ONE SPLIT
# ---------------------------------------------------------------------------
def process(df: pd.DataFrame, name: str, is_train: bool) -> pd.DataFrame:
    print(f"\n>>> {name} ({len(df):,} rows)")
    df = df.copy()
    df["comment_text"] = df["comment_text"].fillna("").astype(str)

    # Dedupe TRAIN only (never touch test integrity)
    if is_train:
        before = len(df)
        subset = ["comment_text"] + [c for c in LABELS if c in df.columns]
        df = df.drop_duplicates(subset=subset, keep="first").reset_index(drop=True)
        if before != len(df):
            print(f"    dropped {before - len(df):,} exact duplicate rows")

    # Feature engineering from RAW text
    feats = build_features(df["comment_text"])

    # Apply BOTH cleaning levels
    print("    cleaning level 1 (light) ...")
    df["comment_light"] = df["comment_text"].progress_apply(clean_light)
    print("    cleaning level 2 (heavy) ...")
    df["comment_heavy"] = df["comment_light"].progress_apply(clean_heavy)

    # Attach features + emptiness flags
    df = pd.concat([df, feats], axis=1)
    df["is_empty_light"] = (df["comment_light"].str.strip() == "").astype(int)
    df["is_empty_heavy"] = (df["comment_heavy"].str.strip() == "").astype(int)
    # If light survives but heavy is empty -> likely non-Latin script / emoji only
    df["is_non_latin"] = (
        (df["is_empty_heavy"] == 1) & (df["is_empty_light"] == 0)
    ).astype(int)

    n_heavy = int(df["is_empty_heavy"].sum())
    n_light = int(df["is_empty_light"].sum())
    n_nonlatin = int(df["is_non_latin"].sum())
    if n_nonlatin:
        print(f"    {n_nonlatin:,} rows empty in HEAVY but fine in LIGHT "
              f"(non-Latin / emoji) -> KEPT, flagged as is_non_latin")
    if is_train and n_light:
        df = df[df["is_empty_light"] == 0].reset_index(drop=True)
        print(f"    dropped {n_light:,} rows with no usable text at all")
    print(f"    {n_heavy:,} rows unusable for bag-of-words -> "
          f"filter on is_empty_heavy for TF-IDF")
    return df


# ---------------------------------------------------------------------------
# K-FOLD SPLIT (multilabel-stratified, group-aware)
# ---------------------------------------------------------------------------
def add_folds(df: pd.DataFrame, n_splits: int = N_FOLDS) -> pd.DataFrame:
    """
    Assign a `fold` column so every downstream model uses IDENTICAL splits.

    Group-aware: rows whose cleaned text collapses to the same string share
    a fold — prevents train/val leakage from near-duplicate comments.
    """
    df = df.copy()

    # Group key: normalized cleaned text; fall back to id for empty cases
    group_key = (
        df["comment_light"].str.lower().str.replace(r"[^a-z0-9]+", "", regex=True)
    )
    df["_group"] = group_key.where(group_key != "", df["id"].astype(str))

    reps = df.drop_duplicates(subset="_group").reset_index(drop=True)
    y = reps[LABELS].values

    if HAS_ITERSTRAT:
        splitter = MultilabelStratifiedKFold(
            n_splits=n_splits, shuffle=True, random_state=SEED
        )
        pairs = splitter.split(reps, y)
        method = "iterative multilabel stratification, group-aware"
    else:
        # Fallback: stratify on the 6-bit label signature ("100010" etc.)
        from sklearn.model_selection import StratifiedKFold
        sig = reps[LABELS].astype(int).astype(str).agg("".join, axis=1)
        counts = sig.value_counts()
        rare = counts[counts < n_splits].index
        sig = sig.where(~sig.isin(rare), "rare")
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        pairs = splitter.split(reps, sig)
        method = "label-signature stratification, group-aware"

    reps["fold"] = -1
    for fold_id, (_, val_idx) in enumerate(pairs):
        reps.loc[reps.index[val_idx], "fold"] = fold_id

    df["fold"] = df["_group"].map(reps.set_index("_group")["fold"])
    n_grouped = len(df) - len(reps)
    df = df.drop(columns="_group")

    print(f"\n    folds: {n_splits} via {method}")
    if n_grouped:
        print(f"    {n_grouped:,} duplicate-text rows pinned to their group's fold "
              f"(prevents train/val leakage)")
    dist = df.groupby("fold")[LABELS].mean() * 100
    print("    positive rate per fold (%) - should be near-identical:")
    print(dist.round(2).to_string())
    return df


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
print(f"Reading from: {INPUT_DIR}")
print(f"Writing to:   {OUTPUT_DIR}")

# --- TRAIN ---
train = pd.read_csv(INPUT_DIR / "train.csv.zip")
audit(train, "train.csv")
train_clean = process(train, "train.csv", is_train=True)
train_clean = add_folds(train_clean, n_splits=N_FOLDS)

train_out = OUTPUT_DIR / "train_clean.parquet"
train_clean.to_parquet(train_out, index=False)
print(f"    wrote {train_out} ({train_out.stat().st_size / 1024 / 1024:.2f} MB, "
      f"{len(train_clean):,} rows, {len(train_clean.columns)} cols)")

# --- TEST ---
test = pd.read_csv(INPUT_DIR / "test.csv.zip")
audit(test, "test.csv")
test_clean = process(test, "test.csv", is_train=False)

test_out = OUTPUT_DIR / "test_clean.parquet"
test_clean.to_parquet(test_out, index=False)
print(f"    wrote {test_out} ({test_out.stat().st_size / 1024 / 1024:.2f} MB, "
      f"{len(test_clean):,} rows, {len(test_clean.columns)} cols)")

# --- TEST LABELS (drop scoring-excluded -1 rows) ---
test_labels = pd.read_csv(INPUT_DIR / "test_labels.csv.zip")
print(f"\n>>> test_labels.csv ({len(test_labels):,} rows)")
scored = (test_labels[LABELS] != -1).all(axis=1)
test_labels_clean = test_labels[scored].reset_index(drop=True)
print(f"    unscored rows (-1) removed : {int((~scored).sum()):,}")
print(f"    usable rows for evaluation : {len(test_labels_clean):,}")
for c in LABELS:
    n = int(test_labels_clean[c].sum())
    print(f"      {c:<14} {n:>7,}  ({n / max(len(test_labels_clean), 1) * 100:5.2f}%)")

tl_out = OUTPUT_DIR / "test_labels_clean.parquet"
test_labels_clean.to_parquet(tl_out, index=False)
print(f"    wrote {tl_out}")

# --- SUMMARY ---
print("\n" + "=" * 60)
print("DONE. Columns you'll use downstream:")
print("  comment_text       raw original, never modified")
print("  comment_light  ->  transformers (BERT / DistilBERT)")
print("  comment_heavy  ->  TF-IDF / classical ML")
print("  fold           ->  every model uses SAME CV splits")
print("  is_empty_heavy ->  filter this=1 rows for classical")
print("  n_you, caps_ratio, has_shouting, n_masked_words, ...")
print("                 ->  hand-crafted features for classical (hstack w/ TF-IDF)")
print("=" * 60)

p95_light = int(train_clean["comment_light"].str.split().str.len().quantile(0.95))
print(f"\nSuggested transformer max_seq_length: {p95_light} tokens (p95 of comment_light)")
print("\nNext step: EDA (co-occurrence heatmap, per-label word clouds, length hist)")